# Semana 6: Apuntes de la clase

**Riesgo y rendimiento en los mercados financieros** (CFA L1, Portfolio Management: *Portfolio Risk and Return: Part I*; puente a CFA L2: *Measuring and Managing Market Risk*)

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JonathanRosasV/topicos-finanzas-upao/blob/main/06_riesgo_rendimiento/clase06_apuntes.ipynb)

Este notebook resume los conceptos que debiste llevarte de la clase, con los ejemplos numéricos ejecutables. Con esta semana arranca la unidad 2 (portafolios): todo lo que viene (Markowitz, CAPM y multifactoriales, evaluación de desempeño) se apoya en las cinco fórmulas de hoy.

## Glosario de siglas de la semana

Antes de los conceptos, el idioma. Estas son las siglas y símbolos que usamos esta semana; los de origen inglés se usan tal cual en la práctica profesional y en el examen CFA.

| Sigla | Significado | En pocas palabras |
|---|---|---|
| HPR | *Holding Period Return* (retorno del periodo de tenencia) | Lo que ganaste entre dos fechas: variación del precio más dividendos, sobre el precio inicial. |
| $E(R)$ | *Expected Return* (retorno esperado) | El promedio de los retornos posibles, ponderado por su probabilidad. |
| $\sigma^2$ | Varianza | Dispersión de los retornos alrededor de su media, en unidades al cuadrado. |
| $\sigma$ | Desviación estándar o volatilidad | La raíz de la varianza: el riesgo en las mismas unidades del retorno. |
| $\sigma_{AB}$ (Cov) | Covarianza | Si dos activos se desvían de su media en la misma dirección (positiva) o en direcciones opuestas (negativa). |
| $\rho$ | Correlación | La covarianza estandarizada, siempre entre $-1$ y $+1$; la que sí se interpreta. |
| $w$ | *Weight* (peso) | Fracción del portafolio invertida en cada activo; los pesos suman 1. |
| PMV | Portafolio de mínima varianza | La mezcla con el menor riesgo posible (en inglés, *minimum variance portfolio*). |
| $U$ y $A$ | Utilidad y coeficiente de aversión al riesgo | $U = E(R) - \tfrac{1}{2}A\sigma^2$: cuánto vale un portafolio para un inversionista que castiga la varianza. |
| CAPM | *Capital Asset Pricing Model* | El modelo de la semana 2: solo se paga el riesgo sistemático, medido por el beta. |
| $\beta$ | Beta | Sensibilidad del activo al mercado: la medida del riesgo sistemático. |
| ETF | *Exchange Traded Fund* | Fondo que cotiza en bolsa y replica un índice; en la práctica usamos SPY (S&P 500) y EPU (MSCI Perú). |
| BVL | Bolsa de Valores de Lima | El mercado local; sus acciones cotizan en soles y en Yahoo llevan el sufijo `.LM`. |

## 1. Medir el rendimiento

$$R_t = \frac{P_t - P_{t-1} + D_t}{P_{t-1}}$$

Con varios periodos hay dos promedios y no responden la misma pregunta. La **media aritmética** estima el retorno esperado de un periodo cualquiera. La **media geométrica** dice a qué tasa creció de verdad el capital:

$$\bar{R}_G = \left[\prod_{t=1}^{T}(1+R_t)\right]^{1/T} - 1$$

La geométrica nunca supera a la aritmética, y la brecha crece con la volatilidad (aproximadamente $\bar{R}_G \approx \bar{R} - \sigma^2/2$).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import numpy as np
from utils.finanzas import (estadisticos_escenarios, covarianza_escenarios, portafolio_dos_activos,
                            peso_minima_varianza, riesgo_equiponderado, anualizar)

# HPR: compras a 50, cobras un dividendo de 1 y vendes a 54
print(f"HPR = {(54 - 50 + 1) / 50:.1%}")

# El ejemplo de la clase: +50% y luego -50%
r = np.array([0.50, -0.50])
print(f"Media aritmetica = {r.mean():.1%} | Media geometrica = {np.prod(1 + r) ** (1 / len(r)) - 1:.1%}")
print(f"Capital final por cada 100 invertidos = {100 * np.prod(1 + r):.0f}")

Salida esperada: HPR de 10.0%; media aritmética 0.0% contra geométrica de $-13.4$%: de 100 pasaste a 75. La volatilidad se come el crecimiento compuesto.

## 2. Retorno esperado y riesgo de un activo

Mirando hacia adelante, con escenarios:

$$E(R) = \sum_s p_s R_s \qquad\qquad \sigma^2 = \sum_s p_s\,[R_s - E(R)]^2$$

Mirando hacia atrás, con una muestra de $T$ retornos, la media muestral y la varianza muestral (con $T-1$ en el denominador; es lo que calcula `pandas` con `.std()`).

Para **anualizar** datos mensuales: $\mu_{anual} = 12\,\mu_{mensual}$ y $\sigma_{anual} = \sqrt{12}\,\sigma_{mensual}$. La media escala con el tiempo y la volatilidad con su raíz (supone retornos sin autocorrelación).

In [ ]:
# Nuestra minera por escenarios: auge, normal, recesion
probs = [0.25, 0.50, 0.25]
minera = [36, 20, -28]            # retornos en %

mu_a, sigma_a = estadisticos_escenarios(probs, minera)
print(f"Minera: E(R) = {mu_a:.1f}% | sigma = {sigma_a:.1f}%")

# Cuanto aporta cada escenario a la varianza
for nombre, p, r in zip(["auge", "normal", "recesion"], probs, minera):
    print(f"  {nombre:9s} aporta {p * (r - mu_a) ** 2:5.0f} de {sigma_a ** 2:.0f}")

# Anualizar: un activo con media mensual de 1% y volatilidad mensual de 6%
mu_anual, sigma_anual = anualizar(0.01, 0.06, periodos=12)
print(f"Anualizado: mu = {mu_anual:.1%} | sigma = {sigma_anual:.1%} (no 72%: se multiplica por raiz de 12)")

Salida esperada: $E(R) = 12\%$ y $\sigma = 24\%$. La recesión sola aporta 400 de los 576 de varianza: el riesgo vive en las colas. El activo mensual anualiza a 12.0% de retorno y 20.8% de volatilidad.

## 3. Covarianza y correlación

$$\sigma_{AB} = \sum_s p_s\,[R_{A,s} - E(R_A)]\,[R_{B,s} - E(R_B)] \qquad\qquad \rho_{AB} = \frac{\sigma_{AB}}{\sigma_A\,\sigma_B}$$

La covarianza da el signo de la relación, pero su magnitud depende de las unidades; la correlación la estandariza entre $-1$ y $+1$. Entre acciones casi todas las correlaciones son positivas (todas cargan el mismo ciclo) y, dato incómodo, suben en las crisis.

Un ejemplo con signo negativo: una aurífera que funciona como refugio (le va mejor cuando a la economía le va peor).

In [ ]:
aurifera = [-4, 8, 20]            # retornos en % en auge, normal, recesion

mu_c, sigma_c = estadisticos_escenarios(probs, aurifera)
cov_ac = covarianza_escenarios(probs, minera, aurifera)
rho_ac = cov_ac / (sigma_a * sigma_c)
print(f"Aurifera: E(R) = {mu_c:.1f}% | sigma = {sigma_c:.2f}%")
print(f"Covarianza minera-aurifera = {cov_ac:.0f} | correlacion = {rho_ac:.2f}")

Salida esperada: covarianza de $-192$ y correlación de $-0.94$. El $-192$ solo no dice nada; el $-0.94$ sí: casi una cobertura perfecta. (Con tres escenarios movidos por un solo factor las correlaciones salen extremas; con datos reales son mucho más moderadas, como verás en la práctica.)

## 4. El portafolio de dos activos

$$E(R_p) = w_A E(R_A) + w_B E(R_B) \qquad\qquad \sigma_p^2 = w_A^2\sigma_A^2 + w_B^2\sigma_B^2 + 2\,w_A w_B\,\rho_{AB}\,\sigma_A\sigma_B$$

El retorno del portafolio es el promedio ponderado; el riesgo no. Solo con $\rho = +1$ la volatilidad del portafolio es el promedio ponderado de las volatilidades; con cualquier $\rho < 1$ es menor. Esa diferencia es la diversificación.

El ejemplo de la clase: minera ($12\%$, $\sigma = 24\%$) y eléctrica regulada ($8\%$, $\sigma = 12\%$), con correlación histórica de 0.25.

In [ ]:
mu_b, sigma_b, rho = 8.0, 12.0, 0.25

print("Portafolio 50/50 segun la correlacion:")
for r_ in [1.0, 0.25, 0.0, -1.0]:
    e_p, s_p = portafolio_dos_activos(0.5, mu_a, mu_b, sigma_a, sigma_b, r_)
    print(f"  rho = {r_:+.2f} -> E(Rp) = {e_p:.1f}% | sigma_p = {s_p:.1f}%")

# Portafolio de minima varianza
w_min = peso_minima_varianza(sigma_a, sigma_b, rho)
e_min, s_min = portafolio_dos_activos(w_min, mu_a, mu_b, sigma_a, sigma_b, rho)
print(f"\nMinima varianza: {w_min:.1%} en la minera -> E(Rp) = {e_min:.1f}% | sigma_p = {s_min:.1f}%")

Salida esperada: con $\rho = 0.25$ el 50/50 tiene $\sigma_p = 14.7\%$ contra un promedio ponderado de 18.0%: 3.3 puntos de riesgo desaparecieron sin tocar el retorno esperado de 10%. El portafolio de mínima varianza pone 12.5% en la minera y tiene $\sigma_p = 11.6\%$, **menos que la eléctrica sola** (12%): agregar un poco del activo más riesgoso bajó el riesgo total y subió el retorno. Las mezclas por debajo de ese punto son ineficientes.

$$w_A^{*} = \frac{\sigma_B^2 - \sigma_{AB}}{\sigma_A^2 + \sigma_B^2 - 2\sigma_{AB}}$$

## 5. Diversificación con n activos y los dos tipos de riesgo

Con pesos iguales, varianza promedio $\bar{\sigma}^2$ y covarianza promedio $\overline{cov}$:

$$\sigma_p^2 = \frac{1}{n}\,\bar{\sigma}^2 + \frac{n-1}{n}\,\overline{cov} \;\;\xrightarrow{\;n\to\infty\;}\;\; \overline{cov}$$

El primer término es el riesgo **idiosincrático** (propio de cada empresa): se diluye al agregar activos. El segundo es el riesgo **sistemático** (de mercado): no desaparece por más activos que sumes. Es el piso.

In [ ]:
# Acciones tipicas: volatilidad media de 30% y correlacion media de 0.30
for n in [1, 2, 5, 10, 30, 10000]:
    print(f"  n = {n:5d} -> sigma_p = {riesgo_equiponderado(n, 30, 0.30):.1f}%")
print(f"Piso sistematico = raiz(0.30) x 30% = {np.sqrt(0.30) * 30:.1f}%")

Salida esperada: 30.0, 24.2, 19.9, 18.2, 17.1 y, en el límite, 16.4%. De 1 a 10 activos el riesgo cae 11.8 puntos; de 10 a infinito, solo 1.8 más.

**La deuda de la semana 2, pagada.** Si eliminar el riesgo idiosincrático es gratis (basta diversificar), el mercado no tiene por qué pagarte por cargarlo: la prima por riesgo compensa solo el sistemático, y eso es lo que mide el beta. Por eso descontamos con $K_e = R_f + \beta \times ERP$ y no con la volatilidad total del activo.

## 6. Aversión al riesgo: cómo se elige

La diversificación define el menú de combinaciones; cuál elegir depende del inversionista. El estándar del CFA es $U = E(R) - \tfrac{1}{2}A\sigma^2$ (retornos en decimales), donde $A$ mide la aversión al riesgo.

In [ ]:
opciones = {"Minera sola": (mu_a, sigma_a), "Electrica sola": (mu_b, sigma_b),
            "50/50": portafolio_dos_activos(0.5, mu_a, mu_b, sigma_a, sigma_b, rho),
            "Minima varianza": (e_min, s_min)}

for A in [4, 1]:
    print(f"Aversion al riesgo A = {A}:")
    for nombre, (e, s) in opciones.items():
        print(f"  {nombre:16s} U = {e / 100 - 0.5 * A * (s / 100) ** 2:.4f}")

Salida esperada: con $A = 4$ gana la mínima varianza (0.0580) y la minera sola es la peor opción (0.0048), a pesar de tener el mayor retorno esperado. Con $A = 1$ el orden se invierte y la minera sola es la preferida (0.0912). El menú es el mismo para todos; la elección, no.

## 7. Respuestas a los ítems de la clase

**1. A.** $(1.25 \times 0.80)^{1/2} - 1 = 0$. La opción B (2.5%) es la media aritmética: el capital no creció nada.

**2. A.** $\sigma_p^2 = 0.6^2(400) + 0.4^2(100) + 0 = 160$, así que $\sigma_p = 12.6\%$. La B (16.0%) es el promedio ponderado, que solo vale con $\rho = +1$; la C olvida los pesos.

**3. C.** El término $\bar{\sigma}^2/n$ desaparece y queda la covarianza promedio: el piso sistemático. La A sería cierta solo si las covarianzas fueran cero.

**4. C.** A menor correlación, mayor beneficio de diversificación; de las tres, $-0.4$ es la menor. Ojo: no hace falta correlación negativa para diversificar, basta que sea menor que 1.

## 8. Lista de verificación

Sabes de esta semana si puedes: calcular un HPR y explicar cuándo usar la media aritmética o la geométrica; obtener $E(R)$ y $\sigma$ por escenarios y con datos históricos, y anualizarlos; distinguir covarianza de correlación; calcular el riesgo de un portafolio de dos activos y su mezcla de mínima varianza; explicar por qué la diversificación tiene un piso y conectar ese piso con el beta del CAPM.